In [391]:
import io
from urllib.parse import parse_qs, urlparse
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import requests
from zipfile import ZipFile

In [392]:
# El link de onedrive.aspx es interno del tenant y redirige a login.microsoftonline.com.
# El que si funciona sin cuenta es el link de compartir publicado en gob.mx/sesnsp:
url_mex = "https://sspcgob-my.sharepoint.com/:u:/g/personal/cni_sspc_gob_mx/IQBBP0Bh8YBEQpdct54JoLbkAfsR_nJJ5RJa03UwlgRmA2E?e=Gca5VL"
url_2015_2025 = "https://sspcgob-my.sharepoint.com/:u:/g/personal/cni_sspc_gob_mx/IQDbKVoKKhp4SrgMtPRM-FILARntV1yA0s8T7nu84A_HF70?e=tZc3uw"


In [393]:
def download_sharepoint_zip(share_url):
    """Abre un link publico de SharePoint y regresa el zip que hay detras."""
    session = requests.Session()
    session.headers["User-Agent"] = "Mozilla/5.0"

    # Esta primera vuelta es la que deja las cookies de invitado y resuelve
    # la ruta real del archivo dentro de la query del visor.
    viewer = session.get(share_url)
    viewer.raise_for_status()
    source_url = parse_qs(urlparse(viewer.url).query)["id"][0]

    download_url = viewer.url.split("?")[0].replace("onedrive.aspx", "download.aspx")
    response = session.get(download_url, params={"SourceUrl": source_url})
    response.raise_for_status()
    return ZipFile(io.BytesIO(response.content))

In [394]:
def read_crime_data(share_url, type_encoding:str):
    """Regresa el CSV de incidencia delictiva que viene dentro del zip."""
    with download_sharepoint_zip(share_url) as archive:
        print(archive.filename)
        name = next(n for n in archive.namelist() if n.lower().endswith(".csv"))
        return pd.read_csv(archive.open(name), encoding=type_encoding)


In [395]:
df = read_crime_data(share_url=url_mex, type_encoding="utf-8")
df.head(10)

None


,Año,Clave_Ent,Entidad,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
0,2026,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,1,1,1,2,2,0,0,2,0,0,0,0
1,2026,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,4,3,2,5,2,0,4,3,0,0,0,0
2,2026,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con otro elemento,0,0,1,1,1,0,1,1,0,0,0,0
3,2026,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,No especificado,0,0,0,0,0,0,1,0,0,0,0,0
4,2026,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Tentativa de homicidio doloso,Tentativa de homicidio doloso,1,0,1,0,0,0,0,0,0,0,0,0
5,2026,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio culposo,Con arma blanca,0,0,0,0,0,0,1,0,0,0,0,0
6,2026,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio culposo,Con arma de fuego,0,0,0,0,0,0,0,0,0,0,0,0
7,2026,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio culposo,Con otro elemento,0,1,0,1,0,0,0,0,0,0,0,0
8,2026,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio culposo,En accidente de tránsito,10,8,11,10,5,11,5,6,0,0,0,0
9,2026,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio culposo,No especificado,1,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
df_2025 = read_crime_data(share_url=url_2015_2025, type_encoding="latin-1")
df_2025.head(5)

None


,Año,Clave_Ent,Entidad,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
0,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,3,0,2,1,1,1,2,1,2,2,2,1
1,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,1,1,0,0,0,1,0,1,0,0,0,1
2,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con otro elemento,0,0,2,2,3,2,0,1,2,0,0,0
3,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,No especificado,2,0,0,1,0,0,0,0,0,0,0,0
4,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio culposo,Con arma de fuego,0,0,0,0,1,0,0,0,0,0,0,0


In [ ]:

def print_columns(dataframe):
    for col in dataframe.columns:
        if dataframe[col].dtype == 'object' or dataframe[col].nunique() < 20:
            print(f"\n{col}: {dataframe[col].nunique()} unique values")
            print(dataframe[col].unique()[:80])  # Show first 10 unique values

print_columns(df)
print('=' * 60)
print_columns(df_2025)


Año: 1 unique values
[2026]

Entidad: 32 unique values
['Aguascalientes' 'Baja California' 'Baja California Sur' 'Campeche'
 'Coahuila de Zaragoza' 'Colima' 'Chiapas' 'Chihuahua' 'Ciudad de México'
 'Durango' 'Guanajuato' 'Guerrero' 'Hidalgo' 'Jalisco' 'México'
 'Michoacán de Ocampo' 'Morelos' 'Nayarit' 'Nuevo León' 'Oaxaca' 'Puebla'
 'Querétaro' 'Quintana Roo' 'San Luis Potosí' 'Sinaloa' 'Sonora' 'Tabasco'
 'Tamaulipas' 'Tlaxcala' 'Veracruz de Ignacio de la Llave' 'Yucatán'
 'Zacatecas']

Bien jurídico afectado: 7 unique values
['La vida y la Integridad corporal' 'Libertad personal'
 'La libertad y la seguridad sexual' 'El patrimonio' 'La familia'
 'La sociedad' 'Otros bienes jurídicos afectados (del fuero común)']

Tipo de delito: 47 unique values
['Homicidio'
 'Otros delitos que atentan contra la vida y la integridad corporal'
 'Lesiones' 'Feminicidio' 'Aborto'
 'Otros delitos que atentan contra la libertad personal' 'Secuestro'
 'Tráfico de menores' 'Rapto' 'Privación ilegal de la

In [415]:
# Printing subtype 
def print_subtype(df1, df2):
    for col in df1.columns:
        if df1[col].name == 'Subtipo de delito':
            for value in df1[col].unique():
                if value not in df2[col].unique():
                    print("Value not found: ", value)

# Equivalente a print_subtype
subtyes_not_df = [y for x in df.columns if x == 'Subtipo de delito' for y in df[x].unique() if y not in data_2025[x].unique()]
print('Subtypes of crimes not in df from 2026: ', len(subtyes_not_df))
print(subtyes_not_df)

Subtypes of crimes not in df from 2026:  30
['Tentativa de homicidio doloso', 'Tentativa de feminicidio', 'Secuestro extorsivo', 'Secuestro con calidad de rehén', 'Secuestro para causar daño', 'Secuestro exprés', 'Privación ilegal de la libertad', 'Retención o sustracción de menores e incapaces', 'Violación a la intimidad sexual', 'Robo de vehículo automotor - Coche de 4 ruedas', 'Robo de vehículo automotor - Motocicleta', 'Robo de vehículo automotor - Embarcaciones', 'Robo de maquinaria - Herramienta industrial o agrícola', 'Robo de maquinaria - Tractores y/o montacargas', 'Robo de maquinaria - Cables, tubos y otros objetos destinados a servicios públicos', 'Extorsión presencial', 'Tentativa de extorsión presencial', 'Extorsión por otros medios', 'Tentativa de extorsión por otros medios', 'Trata de personas con fines de explotación sexual', 'Trata de personas con fines de trabajo o servicios forzados', 'Trata de personas con fines de tráfico de órganos', 'Trata de personas con otros f

In [399]:
df.isnull().sum()

Año                       0
Clave_Ent                 0
Entidad                   0
Bien jurídico afectado    0
Tipo de delito            0
Subtipo de delito         0
Modalidad                 0
Enero                     0
Febrero                   0
Marzo                     0
Abril                     0
Mayo                      0
Junio                     0
Julio                     0
Agosto                    0
Septiembre                0
Octubre                   0
Noviembre                 0
Diciembre                 0
dtype: int64

In [ ]:
df_2025.isnull().sum()

Año                       0
Clave_Ent                 0
Entidad                   0
Bien jurídico afectado    0
Tipo de delito            0
Subtipo de delito         0
Modalidad                 0
Enero                     0
Febrero                   0
Marzo                     0
Abril                     0
Mayo                      0
Junio                     0
Julio                     0
Agosto                    0
Septiembre                0
Octubre                   0
Noviembre                 0
Diciembre                 0
dtype: int64

In [ ]:
grupos = {
    # Diferencia juridica, no criminologica
    'Violación simple': 'violacion',
    'Violación equiparada': 'violacion',

    # Cruza la jerarquia: son dos Tipo de delito distintos
    'Acoso sexual': 'acoso',
    'Hostigamiento sexual': 'acoso',

    # Corte administrativo puro
    'Robo a transeúnte en vía pública': 'robo_transeunte',
    'Robo a transeúnte en espacio abierto al público': 'robo_transeunte',

    'Robo a transportista': 'robo_transportista',
    'Robo a casa habitación': 'robo_casa',

    # Robo vehicular: ver la nota, yo los dejaria separados
    'Robo de vehículo automotor - Coche de 4 ruedas': 'robo_vehiculo',
    'Robo de vehículo automotor - Motocicleta': 'robo_moto',
    'Robo de vehículo automotor - Embarcaciones': 'robo_embarcacion',

    # Aqui NO se juntan: doloso y culposo son procesos distintos
    'Homicidio doloso': 'homicidio_doloso',
    'Homicidio culposo': 'homicidio_culposo',
    'Tentativa de homicidio doloso': 'tentativa_homicidio',

    # Junta el medio (presencial / otros medios), separa consumado de tentativa
    'Extorsión presencial': 'extorsion',
    'Extorsión por otros medios': 'extorsion',
    'Tentativa de extorsión presencial': 'tentativa_extorsion',
    'Tentativa de extorsión por otros medios': 'tentativa_extorsion',

    # Los 4 secuestros son lo mismo operativamente, y ademas son raros
    'Secuestro extorsivo': 'secuestro',
    'Secuestro con calidad de rehén': 'secuestro',
    'Secuestro para causar daño': 'secuestro',
    'Secuestro exprés': 'secuestro',
}


In [401]:
entidad_crimen = df.groupby(['Entidad', 'Tipo de delito', 'Subtipo de delito']).sum().reset_index()
entidad_crimen

,Entidad,Tipo de delito,Subtipo de delito,Año,Clave_Ent,Bien jurídico afectado,Modalidad,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
0,Aguascalientes,Aborto,Aborto,2026,1,La vida y la Integridad corporal,Aborto,1,1,0,0,3,0,2,1,0,0,0,0
1,Aguascalientes,Abuso de confianza,Abuso de confianza,2026,1,El patrimonio,Abuso de confianza,71,64,72,50,53,54,74,49,0,0,0,0
2,Aguascalientes,Abuso sexual,Abuso sexual,2026,1,La libertad y la seguridad sexual,Abuso sexual,0,0,48,55,60,55,44,44,0,0,0,0
3,Aguascalientes,Acoso sexual,Acoso sexual,2026,1,La libertad y la seguridad sexual,Acoso sexual,0,0,0,0,0,0,0,0,0,0,0,0
4,Aguascalientes,Allanamiento de morada,Allanamiento de morada,2026,1,Otros bienes jurídicos afectados (del fuero co...,Allanamiento de morada,32,42,49,36,39,30,33,33,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2523,Zacatecas,Violación,Violación equiparada,2026,32,La libertad y la seguridad sexual,Violación equiparada,9,9,15,15,17,15,12,9,0,0,0,0
2524,Zacatecas,Violación,Violación simple,2026,32,La libertad y la seguridad sexual,Violación simple,10,12,17,14,17,19,21,10,0,0,0,0
2525,Zacatecas,Violación a la intimidad sexual,Violación a la intimidad sexual,2026,32,La libertad y la seguridad sexual,Violación a la intimidad sexual,17,21,19,16,19,16,19,19,0,0,0,0
2526,Zacatecas,Violencia de género en todas sus modalidades d...,Violencia de género en todas sus modalidades d...,2026,32,La familia,Violencia de género en todas sus modalidades d...,0,0,0,0,0,0,0,0,0,0,0,0


In [402]:
entidad_crimen['total'] = entidad_crimen.iloc[:, -12:].sum(axis=1)
entidad_crimen

,Entidad,Tipo de delito,Subtipo de delito,Año,Clave_Ent,Bien jurídico afectado,Modalidad,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre,total
0,Aguascalientes,Aborto,Aborto,2026,1,La vida y la Integridad corporal,Aborto,1,1,0,0,3,0,2,1,0,0,0,0,8
1,Aguascalientes,Abuso de confianza,Abuso de confianza,2026,1,El patrimonio,Abuso de confianza,71,64,72,50,53,54,74,49,0,0,0,0,487
2,Aguascalientes,Abuso sexual,Abuso sexual,2026,1,La libertad y la seguridad sexual,Abuso sexual,0,0,48,55,60,55,44,44,0,0,0,0,306
3,Aguascalientes,Acoso sexual,Acoso sexual,2026,1,La libertad y la seguridad sexual,Acoso sexual,0,0,0,0,0,0,0,0,0,0,0,0,0
4,Aguascalientes,Allanamiento de morada,Allanamiento de morada,2026,1,Otros bienes jurídicos afectados (del fuero co...,Allanamiento de morada,32,42,49,36,39,30,33,33,0,0,0,0,294
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2523,Zacatecas,Violación,Violación equiparada,2026,32,La libertad y la seguridad sexual,Violación equiparada,9,9,15,15,17,15,12,9,0,0,0,0,101
2524,Zacatecas,Violación,Violación simple,2026,32,La libertad y la seguridad sexual,Violación simple,10,12,17,14,17,19,21,10,0,0,0,0,120
2525,Zacatecas,Violación a la intimidad sexual,Violación a la intimidad sexual,2026,32,La libertad y la seguridad sexual,Violación a la intimidad sexual,17,21,19,16,19,16,19,19,0,0,0,0,146
2526,Zacatecas,Violencia de género en todas sus modalidades d...,Violencia de género en todas sus modalidades d...,2026,32,La familia,Violencia de género en todas sus modalidades d...,0,0,0,0,0,0,0,0,0,0,0,0,0


In [403]:
# Esta es una forma de concatenear las columnas 
#new_df = pd.concat([entidad_crimen.iloc[:, :8], entidad_crimen.iloc[:, -1:]], axis=1)

# Esto deberia de dejarme con las columnas que si quiero
new_df = entidad_crimen.drop(entidad_crimen.iloc[:, 7:-1], axis=1)
new_df

,Entidad,Tipo de delito,Subtipo de delito,Año,Clave_Ent,Bien jurídico afectado,Modalidad,total
0,Aguascalientes,Aborto,Aborto,2026,1,La vida y la Integridad corporal,Aborto,8
1,Aguascalientes,Abuso de confianza,Abuso de confianza,2026,1,El patrimonio,Abuso de confianza,487
2,Aguascalientes,Abuso sexual,Abuso sexual,2026,1,La libertad y la seguridad sexual,Abuso sexual,306
3,Aguascalientes,Acoso sexual,Acoso sexual,2026,1,La libertad y la seguridad sexual,Acoso sexual,0
4,Aguascalientes,Allanamiento de morada,Allanamiento de morada,2026,1,Otros bienes jurídicos afectados (del fuero co...,Allanamiento de morada,294
...,...,...,...,...,...,...,...,...
2523,Zacatecas,Violación,Violación equiparada,2026,32,La libertad y la seguridad sexual,Violación equiparada,101
2524,Zacatecas,Violación,Violación simple,2026,32,La libertad y la seguridad sexual,Violación simple,120
2525,Zacatecas,Violación a la intimidad sexual,Violación a la intimidad sexual,2026,32,La libertad y la seguridad sexual,Violación a la intimidad sexual,146
2526,Zacatecas,Violencia de género en todas sus modalidades d...,Violencia de género en todas sus modalidades d...,2026,32,La familia,Violencia de género en todas sus modalidades d...,0


In [404]:
# Renombrar columnas 
name_columns = {
    'Entidad': 'state', 
    'Tipo de delito':'crime', 
    'Subtipo de delito':'subtype', 
    'Año':'year', 
    'Clave_Ent':'code',
    'Modalidad': 'modus',
    }
new_df = new_df.rename(columns=name_columns)
clean = new_df.drop(columns={'Bien jurídico afectado'}, axis=1)
clean

,state,crime,subtype,year,code,modus,total
0,Aguascalientes,Aborto,Aborto,2026,1,Aborto,8
1,Aguascalientes,Abuso de confianza,Abuso de confianza,2026,1,Abuso de confianza,487
2,Aguascalientes,Abuso sexual,Abuso sexual,2026,1,Abuso sexual,306
3,Aguascalientes,Acoso sexual,Acoso sexual,2026,1,Acoso sexual,0
4,Aguascalientes,Allanamiento de morada,Allanamiento de morada,2026,1,Allanamiento de morada,294
...,...,...,...,...,...,...,...
2523,Zacatecas,Violación,Violación equiparada,2026,32,Violación equiparada,101
2524,Zacatecas,Violación,Violación simple,2026,32,Violación simple,120
2525,Zacatecas,Violación a la intimidad sexual,Violación a la intimidad sexual,2026,32,Violación a la intimidad sexual,146
2526,Zacatecas,Violencia de género en todas sus modalidades d...,Violencia de género en todas sus modalidades d...,2026,32,Violencia de género en todas sus modalidades d...,0


In [405]:
total_crimes = clean.groupby('crime')['total'].sum().reset_index()
total_crimes

,crime,total
0,Aborto,636
1,Abuso de confianza,20301
2,Abuso sexual,22099
3,Acoso sexual,7201
4,Allanamiento de morada,9684
5,Amenazas,99495
6,Contra el medio ambiente,2022
7,Corrupción de menores,2342
8,Daño a la propiedad,94293
9,Delitos cometidos por servidores públicos,13359


In [406]:
total_sub = clean.groupby('subtype')['total'].sum().reset_index()
total_sub

,subtype,total
0,Aborto,636
1,Abuso de confianza,20301
2,Abuso sexual,22099
3,Acoso sexual,7201
4,Allanamiento de morada,9684
...,...,...
74,Violación a la intimidad sexual,5270
75,Violación equiparada,4304
76,Violación simple,9319
77,Violencia de género en todas sus modalidades d...,3157


In [407]:
from pathlib import Path  # mover a la celda de imports cuando quieras

# Poblacion anual por entidad: es la serie de CONAPO (conciliacion demografica 2023),
# la misma que el SESNSP usa para sus tasas por cada 100 mil habitantes.
# INEGI solo publica cortes censales (intercensal 2015, censo 2020), no la serie anual.
#
# El zip oficial vive aqui, pero el servidor de CONAPO se cae seguido:
url_conapo_zip = "https://conapo.segob.gob.mx/work/models/CONAPO/pry23/DB/ConDem50a19_ProyPob20a70.zip"

# Este espejo en GitHub sirve el mismo csv por entidad, edad y sexo, y si responde:
url_poblacion = "https://raw.githubusercontent.com/lapanquecita/poblacion-estimada/main/data/estatal.csv"

# Son 24 MB, asi que se guardan en disco la primera vez y de ahi se leen.
cache_poblacion = Path("../../data/mexico/conapo_poblacion_estatal.csv")

In [408]:
def download_population(url=url_poblacion, cache=cache_poblacion):
    """Baja el csv crudo de CONAPO y lo cachea; en corridas siguientes lee el disco."""
    if cache.exists():
        return pd.read_csv(cache)

    raw = pd.read_csv(url)
    cache.parent.mkdir(parents=True, exist_ok=True)
    raw.to_csv(cache, index=False, encoding="utf-8")
    return raw

In [409]:
def read_population(years=range(2015, 2027), **kwargs):
    """Regresa la poblacion total por entidad y anio, con los mismos nombres que clean."""
    raw = download_population(**kwargs)

    # El crudo viene desagregado por edad (0 a 109) y sexo, hay que colapsarlo.
    anual = (
        raw[raw["AÑO"].isin(years)]
        .groupby(["AÑO", "CVE_GEO", "ENTIDAD"], as_index=False)["POBLACION"]
        .sum()
    )

    # CVE_GEO es la misma clave que Clave_Ent del SESNSP, por eso el join va por code
    # y no por nombre: CONAPO dice "Coahuila" donde el SESNSP dice "Coahuila de Zaragoza".
    nombres = {
        "AÑO": "year",
        "CVE_GEO": "code",
        "ENTIDAD": "state_conapo",
        "POBLACION": "population",
    }
    return anual.rename(columns=nombres).sort_values(["year", "code"], ignore_index=True)

In [410]:
pob = read_population()
pob

,year,code,state_conapo,population
0,2015,1,Aguascalientes,1344649
1,2015,2,Baja California,3406162
2,2015,3,Baja California Sur,730405
3,2015,4,Campeche,917075
4,2015,5,Coahuila,3017320
...,...,...,...,...
379,2026,28,Tamaulipas,3777472
380,2026,29,Tlaxcala,1464515
381,2026,30,Veracruz,8111813
382,2026,31,Yucatán,2542536


In [411]:
mex_population = pob.loc[pob['year'] >= 2026]
mex_population 

,year,code,state_conapo,population
352,2026,1,Aguascalientes,1567559
353,2026,2,Baja California,4189285
354,2026,3,Baja California Sur,921734
355,2026,4,Campeche,962014
356,2026,5,Coahuila,3440385
357,2026,6,Colima,771538
358,2026,7,Chiapas,6185469
359,2026,8,Chihuahua,4087306
360,2026,9,Ciudad de México,9159966
361,2026,10,Durango,1939482


In [412]:
new_crime_df = pd.merge(clean, mex_population, on='code', how='left')
new_crime_df

,state,crime,subtype,year_x,code,modus,total,year_y,state_conapo,population
0,Aguascalientes,Aborto,Aborto,2026,1,Aborto,8,2026.0,Aguascalientes,1567559.0
1,Aguascalientes,Abuso de confianza,Abuso de confianza,2026,1,Abuso de confianza,487,2026.0,Aguascalientes,1567559.0
2,Aguascalientes,Abuso sexual,Abuso sexual,2026,1,Abuso sexual,306,2026.0,Aguascalientes,1567559.0
3,Aguascalientes,Acoso sexual,Acoso sexual,2026,1,Acoso sexual,0,2026.0,Aguascalientes,1567559.0
4,Aguascalientes,Allanamiento de morada,Allanamiento de morada,2026,1,Allanamiento de morada,294,2026.0,Aguascalientes,1567559.0
...,...,...,...,...,...,...,...,...,...,...
2523,Zacatecas,Violación,Violación equiparada,2026,32,Violación equiparada,101,2026.0,Zacatecas,1726898.0
2524,Zacatecas,Violación,Violación simple,2026,32,Violación simple,120,2026.0,Zacatecas,1726898.0
2525,Zacatecas,Violación a la intimidad sexual,Violación a la intimidad sexual,2026,32,Violación a la intimidad sexual,146,2026.0,Zacatecas,1726898.0
2526,Zacatecas,Violencia de género en todas sus modalidades d...,Violencia de género en todas sus modalidades d...,2026,32,Violencia de género en todas sus modalidades d...,0,2026.0,Zacatecas,1726898.0


In [413]:
new_crime_df = new_crime_df.drop(columns={'year_x', 'year_y', 'modus','state_conapo'})
new_crime_df

,state,crime,subtype,code,total,population
0,Aguascalientes,Aborto,Aborto,1,8,1567559.0
1,Aguascalientes,Abuso de confianza,Abuso de confianza,1,487,1567559.0
2,Aguascalientes,Abuso sexual,Abuso sexual,1,306,1567559.0
3,Aguascalientes,Acoso sexual,Acoso sexual,1,0,1567559.0
4,Aguascalientes,Allanamiento de morada,Allanamiento de morada,1,294,1567559.0
...,...,...,...,...,...,...
2523,Zacatecas,Violación,Violación equiparada,32,101,1726898.0
2524,Zacatecas,Violación,Violación simple,32,120,1726898.0
2525,Zacatecas,Violación a la intimidad sexual,Violación a la intimidad sexual,32,146,1726898.0
2526,Zacatecas,Violencia de género en todas sus modalidades d...,Violencia de género en todas sus modalidades d...,32,0,1726898.0
